# `train_mlp.py`

按功能分段；每段前有标题，方便查找和修改。

### Imports & constants

In [1]:
# train_mlp.py

import os
import random
import sys

# notebook 没有 __file__：同时把「当前目录」和「上一级目录」加入 sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
for _p in (os.getcwd(), PROJECT_ROOT):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
import torch

from neural_config import *
from neural_data import (
    load_model_data,
    make_expanding_splits,
    make_fold_datasets,
)
from neural_models import MLPDirect
from neural_backtest import (
    predict_dataset,
    build_daily_portfolio,
    rescale_to_target_vol,
    summary_table,
)
from neural_search import fit_ensemble

### Reproducibility & device

In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cuda


### Load data & build expanding-window splits

In [4]:
df = load_model_data(DATA_PATH)

splits = make_expanding_splits(df)

all_predictions = []
search_logs = []

Loaded 240,991 rows | 51 tickers | 2002-04-08 -> 2026-09-02


### Walk-forward training loop (one model per fold)

In [5]:
for fold, split in enumerate(splits, 1):

    print("\n" + "=" * 70)

    print(
        f"FOLD {fold}: "
        f"train <= {split['train_end']} | "
        f"test {split['test_start']}-{split['test_end']}"
    )

    train_ds, val_ds, test_ds = make_fold_datasets(
        df, split, lookback=MLP_LOOKBACK, validation_fraction=VAL_FRACTION
    )

    print(
        f"train={len(train_ds):,} "
        f"val={len(val_ds):,} "
        f"test={len(test_ds):,}"
    )

    def build_model(cfg):
        return MLPDirect(n_features=len(FEATURES), lookback=MLP_LOOKBACK, hidden_size=cfg["hidden_size"], dropout=cfg["dropout"])

    models, search_log = fit_ensemble(
        build_model,
        train_ds,
        val_ds,
        device=device,
        fold=fold,
        tag="MLP",
        search_iter=SEARCH_ITER,
    )

    search_logs.append(search_log)

    predictions = predict_dataset(
        models,
        test_ds,
        device,
    )

    predictions["fold"] = fold

    all_predictions.append(predictions)


FOLD 1: train <= 2006 | test 2007-2011
train=16,766 val=2,040 test=38,748
[MLP fold 1] search 1/20 val_sharpe=+2.055 train_sharpe=+0.584 epoch=30 | {'hidden_size': 10, 'dropout': 0.2, 'batch_size': 1024, 'learning_rate': 0.0001, 'max_grad_norm': 1.0, 'weight_decay': 0.0001}
[MLP fold 1] search 2/20 val_sharpe=+1.723 train_sharpe=+0.473 epoch=30 | {'hidden_size': 40, 'dropout': 0.4, 'batch_size': 1024, 'learning_rate': 0.0001, 'max_grad_norm': 1.0, 'weight_decay': 0.001}
[MLP fold 1] search 3/20 val_sharpe=+1.870 train_sharpe=+0.473 epoch=15 | {'hidden_size': 10, 'dropout': 0.3, 'batch_size': 2048, 'learning_rate': 0.0001, 'max_grad_norm': 10.0, 'weight_decay': 0.001}
[MLP fold 1] search 4/20 val_sharpe=+1.974 train_sharpe=+0.644 epoch=3 | {'hidden_size': 20, 'dropout': 0.1, 'batch_size': 512, 'learning_rate': 0.001, 'max_grad_norm': 1.0, 'weight_decay': 0.001}
[MLP fold 1] search 5/20 val_sharpe=+1.985 train_sharpe=+0.441 epoch=18 | {'hidden_size': 80, 'dropout': 0.4, 'batch_size': 20

### Aggregate OOS predictions

In [6]:
pred = pd.concat(
    all_predictions,
    ignore_index=True,
)

pd.concat(search_logs, ignore_index=True).to_csv(
    "mlp_search_log.csv",
    index=False,
)

pred.to_csv(
    "mlp_oos_predictions.csv",
    index=False,
)

### Daily portfolio returns & performance report

In [7]:
daily = build_daily_portfolio(pred)

raw_ret = daily["port_ret"].dropna()

rescaled_ret = rescale_to_target_vol(
    raw_ret
).dropna()


print("\n=== MLP DIRECT SHARPE — RAW ===")

for k, v in summary_table(raw_ret).items():
    print(f"{k:<20s} {v:.4f}")


print("\n=== MLP DIRECT SHARPE — 15% PORTFOLIO VOL ===")

for k, v in summary_table(rescaled_ret).items():
    print(f"{k:<20s} {v:.4f}")


daily["raw_return"] = raw_ret

daily["rescaled_return"] = rescaled_ret

daily.to_csv(
    "mlp_oos_daily_returns.csv"
)

print("\nSaved:")
print("  mlp_search_log.csv")
print("  mlp_oos_predictions.csv")
print("  mlp_oos_daily_returns.csv")


=== MLP DIRECT SHARPE — RAW ===
E[Return]            0.0233
Vol.                 0.0388
Downside Deviation   0.0331
MDD                  0.0980
Sharpe               0.6021
Sortino              0.7049
Calmar               0.2382
% +ve Returns        0.5479
Ave.P/Ave.L          0.9249

=== MLP DIRECT SHARPE — 15% PORTFOLIO VOL ===
E[Return]            0.1005
Vol.                 0.1591
Downside Deviation   0.1342
MDD                  0.3580
Sharpe               0.6320
Sortino              0.7494
Calmar               0.2808
% +ve Returns        0.5480
Ave.P/Ave.L          0.9269

Saved:
  mlp_search_log.csv
  mlp_oos_predictions.csv
  mlp_oos_daily_returns.csv
